In [27]:
#1. IMPORTANDO BIBLIOTECAS E MÓDULOS

from __future__ import annotations
# Faz o Python tratar anotações de tipo de forma mais flexível. 
# Ajuda quando você usa type hints e quer evitar alguns problemas de referência antecipada. 
# Para teu caso, não é obrigatório, mas deixa o código mais moderno e estável.
#=========================================================================================

from pathlib import Path
# Serve para trabalhar com caminhos de arquivos e pastas de forma mais limpa do que usar 
# strings soltas. Exemplo: checar se o arquivo IFC existe.
#=========================================================================================

from typing import Any, Dict, List, Optional
# É para tipagem.

# Any: qualquer tipo
# Dict: dicionário
# List: lista
# Optional: valor que pode existir ou ser None

# Isso melhora leitura e organização, principalmente num código que vai 
# manipular muitas estruturas.
#=========================================================================================

import ifcopenshell
from ifcopenshell import geom

# É a biblioteca principal para abrir e ler o IFC. Sem ela, o projeto basicamente não anda.
#=========================================================================================

import math
# Traz a biblioteca matemática padrão do Python. Neste projeto, está calculando a distância entre elementos,
# comparação geométrica entre suportes e vigas, avaliação de extensões, offsets e tolerâncias,
# e cálculo com coordenadas 3D.
#=========================================================================================

import time
# Traz a biblioteca padrão do Python para controle de tempo e medições temporais.
# Usada para medir, por exemplo, o tempo da leitura do ifc, tempo da checagem de compliance, etc.
#=========================================================================================

from collections import Counter
# A classe Counter é usada para contar ocorrências de forma rápida e organizada, trabalhando
# na sumarização quantitativa dos resultados, objetivando obter indicadores
# simples e legíveis.
#=========================================================================================

from collections import defaultdict
# Serve para criar dicionários que já têm um valor padrão automático para chaves
# que ainda não existem. Ou seja, evita ter que fazer toda hora algo como:
# if chave not in dicionario:
#   dicionario[chave] = []
# Aplica-se, neste trabalho, por exemplo, depois de encontrar vários pares unistrut-beam,
# agrupando todas as ocorrências por beam_id, ou todos os suportes ligados a uma mesma
# viga, ou todas as falhas por tipo, etc.
#=========================================================================================

import pandas as pd
import os

In [ ]:
#Parametros de entrada

# ajuste o caminho se necessário
ifc_path = r"MAIC_ROOM_02.ifc" 

# ajuste o caminho se necessário
ifc_file = "MAIC_ROOM_02.ifc"   

# Substitua o texto "sua chave aqui", entre aspas, por sua chave Anthropic
os.environ["ANTHROPIC_API_KEY"] = "sua chave aqui"

#Coloque suas perguntas a seguir
user_question = (
    "Quais unistruts estão em FAIL e quais são os motivos principais? "
    "Apresente em formato de tabela com: Tag do Unistrut, Tag da Viga, "
    "Overhang Mínimo (mm) e Motivo da Falha,"
    "Quantas cantilever estão em falha?"
    "Quais as salas de cada unistrut?"
)

In [29]:
# 2. DEFININDO AS FUNÇÕES PARA LER O ARQUIVO IFC E FILTRAR OS ELEMENTOS DE INTERESSE

def safe_str(value: Any) -> str:

    if value is None:
        return ""
    try:
        return str(value)
    except Exception:
        return ""
    # Converte qualquer valor para string sem quebrar o código. Se vier None ou algum tipo estranho, retorna string vazia.
    # No IFC, vários campos podem vir como None ou outros objetos que nem sempre convertem bem para string. 
    # Esta função evita erro em operações simples de leitura e comparação textual. Funciona como uma camada defensiva:
    # Antes de comparar nomes, tags, descrições... o código garante que esteja em formato string
#=====================================================================================================================

def get_attr(entity: Any, attr_name: str, default: str = "") -> str:

    try:
        value = getattr(entity, attr_name, None)
        if value is None:
            return default
        return safe_str(value)
    except Exception:
        return default
    
    # Lê atributos diretamente de uma entidade IFC com segurança. Ex.: Name, Description, Tag, ObjectType, GlobalId.
    # Acontece que uma entidade IFC pode ou não ter certos atributos preenchidos. Essa função lê tais atributos
    # sem estourar erro caso o atributo não exista ou venha vazio.
#=====================================================================================================================

def get_psets(entity: Any) -> Dict[str, Dict[str, Any]]:
    psets: Dict[str, Dict[str, Any]] = {}

    try:
        rels = getattr(entity, "IsDefinedBy", []) or []
    except Exception:
        rels = []

    for rel in rels:
        try:
            if not rel.is_a("IfcRelDefinesByProperties"):
                continue

            prop_def = rel.RelatingPropertyDefinition
            if prop_def is None or not prop_def.is_a("IfcPropertySet"):
                continue

            pset_name = safe_str(getattr(prop_def, "Name", "UnnamedPset"))
            psets[pset_name] = {}

            for prop in getattr(prop_def, "HasProperties", []) or []:
                if not prop.is_a("IfcPropertySingleValue"):
                    continue

                prop_name = safe_str(getattr(prop, "Name", "UnnamedProperty"))
                nominal = getattr(prop, "NominalValue", None)

                if nominal is not None and hasattr(nominal, "wrappedValue"):
                    value = nominal.wrappedValue
                else:
                    value = nominal

                psets[pset_name][prop_name] = value

        except Exception:
            continue

    return psets
    # Extrai Property Sets de uma entidade IFC e organiza-os em uma estrutura de dicionário Python
    # Muitos elementos proxy não são bem identificáveis pelos atributos principais. Frequentemente,
    # a informação decisiva está dentro dos Psets, especialmente propriedades customizadas exportadas do Revit
    # ou de outro ambiente BIM. Essencial para localizar propriedades como STS_Name e identificadores de tipo.
#=====================================================================================================================

def get_sts_name(entity: Any) -> Optional[str]:

    psets = get_psets(entity)

    for _, props in psets.items():
        for prop_name, value in props.items():
            if safe_str(prop_name).strip().lower() == "sts_name":
                clean_value = safe_str(value).strip()
                return clean_value if clean_value else None

    return None
    # Procura especialmente a propriedade STS_Name dentro dos Psets da entidade e retorna seu valor.
#=====================================================================================================================

def flatten_text_fields(entity: Any, psets: Dict[str, Dict[str, Any]]) -> str:
    """
    Junta vários campos de texto da entidade e dos Psets em uma única string,
    para facilitar buscas por palavras-chave.
    """
    parts: List[str] = []

    for attr in ("Name", "ObjectType", "Description", "Tag", "GlobalId"):
        parts.append(get_attr(entity, attr))

    try:
        for rel in getattr(entity, "IsTypedBy", []) or []:
            type_obj = getattr(rel, "RelatingType", None)
            if type_obj:
                parts.append(get_attr(type_obj, "Name"))
                parts.append(get_attr(type_obj, "Description"))
    except Exception:
        pass

    for pset_name, props in psets.items():
        parts.append(pset_name)
        for key, value in props.items():
            parts.append(safe_str(key))
            parts.append(safe_str(value))

    return " | ".join([p for p in parts if p]).lower()
    # Junta vários campos textuais da entidade e dos Property Sets em uma única string consolidada. Nem sempre o modelo está bem padronizado.
    # Às vezes, a informação relevante está no Name, ou no Description, ou no ObjectType, ou no nome do tipo, etc.
    # Essa função configura uma espécie de texto consolidado pesquisável. Quando o STS_Name não existe ou não ajuda, o código procura termos relevantes
    # dentro desse texto unificado.
#=====================================================================================================================

def looks_like_target_proxy(entity: Any, psets: Dict[str, Dict[str, Any]]) -> Optional[str]:

    sts_name = get_sts_name(entity)

    if sts_name:
        key = sts_name.strip().lower()

        sts_mapping = {
            "trapeze": "TRAPEZE",
            "single unistrut": "UNISTRUT",
            "double unistrut": "UNISTRUT",
            "cantilever": "CANTILEVER",
            "cantelever": "CANTILEVER",
        }

        if key in sts_mapping:
            return sts_mapping[key]

    text = flatten_text_fields(entity, psets)

    unistrut_keywords = [
        "single unistrut",
        "double unistrut",
        "sts_structuralframing_single_p1000",
        "p1000 (single unistrut)",
    ]

    cantilever_keywords = [
        "cantilever",
        "cantelever",
        "p1000 (cantilever)",
        "p1000 (cantelever)",
        "sts_structuralframing_cantelever",
    ]

    trapeze_keywords = [
        "trapeze",
    ]

    if any(k in text for k in trapeze_keywords):
        return "TRAPEZE"

    if any(k in text for k in unistrut_keywords):
        return "UNISTRUT"

    if any(k in text for k in cantilever_keywords):
        return "CANTILEVER"

    return None
    # Serve para classificar se um proxy IFC é ser um dos suportes-alvo do projeto. Ele tenta reconhecer: Trapeze, unistrut, cantilever
    # ou retorna None quanto o elemento não parece ser um suporte de interesse. Basicamente, é aqui que ocorre a triagem dos proxies, afinal,
    # o modelo pode ter muitos IfcBuildingElementProxy, mas só alguns deles são relevantes para a regra de compliance.
#=====================================================================================================================

def summarize_entity(entity: Any) -> Dict[str, Any]:

    return {
        "step_id": entity.id(),
        "global_id": get_attr(entity, "GlobalId"),
        "class": entity.is_a(),
        "name": get_attr(entity, "Name"),
        "object_type": get_attr(entity, "ObjectType"),
        "tag": get_attr(entity, "Tag"),
        "description": get_attr(entity, "Description"),
        "sts_name": get_sts_name(entity),
    }
    # Retorna um resumo simples e estruturado da entidade IFC. Depois de filtrar oe slementos relevantes, cria uma representação resumida
    # com informações básicas. Assim, não é necessário carregar a entidade IFC inteira, gerando somente uma entidade compacta e intepretável,
    # ideal para reutilização da informação em passos seguintes e para documentaçào dos resultados.

In [30]:
#3. FAZENDO A CHECAGEM DO CASO 01 - UNISTRUTS X VIGAS - PAREAMENTO POR PROJEÇÃO + ORIENTAÇÃO

def get_shape_vertices_xyz(element):
    try:
        settings = geom.settings()
        settings.set("use-world-coords", True)
        settings.set("convert-back-units", True)

        shape = geom.create_shape(settings, element)
        verts = shape.geometry.verts

        if not verts or len(verts) < 3:
            return None

        xs = verts[0::3]
        ys = verts[1::3]
        zs = verts[2::3]

        return list(zip(xs, ys, zs))
    except Exception as e:
        print(f"Erro ao obter vértices XYZ do elemento {element.id()} - {get_attr(element, 'Name')}: {e}")
        return None
    # Extrai os vértices geométricos 3D de um elemento IFC. Ela usa ifcopenshell.geom.create_shape para converter a 
    # entidade IFC em geometria analisável. Depois, pega os vértices dessa geometria e os organiza como uma lista de pontos (x,y,z).
#=====================================================================================================================
    
def get_bbox_and_center_from_points(points_xyz):
    xs = [p[0] for p in points_xyz]
    ys = [p[1] for p in points_xyz]
    zs = [p[2] for p in points_xyz]

    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    zmin, zmax = min(zs), max(zs)

    return {
        "xmin": xmin,
        "ymin": ymin,
        "zmin": zmin,
        "xmax": xmax,
        "ymax": ymax,
        "zmax": zmax,
        "cx": (xmin + xmax) / 2,
        "cy": (ymin + ymax) / 2,
        "cz": (zmin + zmax) / 2,
    }
    # Calcula a bounding box e o centro geométrico do elemento a partir dos seus pontos 3D. Basicamente, depois que os
    # vértices são extraídos, essa função encontra o mínimo e máximo em X, Y, Z e os centros nas três dimensões.
    # Ela transforma uma nuvem de pontos em uma representação geométrica compacta, ideal para comparações rápidas entre
    # elementos.
#=====================================================================================================================

def bbox_sizes(bbox: dict) -> dict:
    return {
        "sx": bbox["xmax"] - bbox["xmin"],
        "sy": bbox["ymax"] - bbox["ymin"],
        "sz": bbox["zmax"] - bbox["zmin"],
    }
    # Calcula as dimensões da bounding box em cada eixo, para registrar o porte geométrico
    # dos elementos.
#=====================================================================================================================

def get_unistrut_entities(model):
    proxies = model.by_type("IfcBuildingElementProxy")
    unistruts = []

    for proxy in proxies:
        psets = get_psets(proxy)
        kind = looks_like_target_proxy(proxy, psets)
        if kind == "UNISTRUT":
            unistruts.append(proxy)

    return unistruts
    # Filtra, dentro do modelo IFC, quais IfcBuildingElementProxy parecem ser unistruts.
    # Basicamente, prepara a lista de candidatos que entrarão na análise geométrica.
#=====================================================================================================================

def get_main_axis_from_bbox(bbox: dict) -> str:
    dx = bbox["xmax"] - bbox["xmin"]
    dy = bbox["ymax"] - bbox["ymin"]
    return "X" if dx >= dy else "Y"
    # Infere o eixo predominante do elemento no plano XY. Permite dizer se uma viga
    # e um unistrut são paralelos ou não.
#=====================================================================================================================

def classify_relative_orientation(axis_a: str, axis_b: str) -> str:
    if axis_a == axis_b:
        return "PARALLEL"
    return "PERPENDICULAR"
    # Classifica a orientação relativa entre dois elementos como paralela ou perpendicular,
    # para, em seguida, determinar a lógica de pareamento que será aplicada.
#=====================================================================================================================

def axis_gap(min1, max1, min2, max2) -> float:
    if max1 < min2:
        return min2 - max1
    elif max2 < min1:
        return min1 - max2
    else:
        return 0.0
    # Calcula a distância entre dois intervalos em um eixo. Pode ser usada para comparar
    # intervalos em qualquer das 3 dimensões. Se os intervalos se sobrepõem, reotrna 0.
    # Caso contrário, reotrna a folga entre eles.
#=====================================================================================================================

def interval_overlap_length(min1, max1, min2, max2) -> float:
    return max(0.0, min(max1, max2) - max(min1, min2))
    # Calcula o comprimento de sobreposição entre dois intervalos, para decidir se
    # a relação geometrica é suficientemente forte para considerar um pareamento válido.
#=====================================================================================================================

def point_to_interval_gap(point, minv, maxv) -> float:
    if point < minv:
        return minv - point
    if point > maxv:
        return point - maxv
    return 0.0
    # Calcula a distância entre um ponto e um intervalo. Entra especialmente no caso 
    # perpendicular, para verificar quão "dentro" ou "fora" um centro está em relação ao
    # alcance do outro elemento. É usada para compor cross_miss_x e cross_miss_y que medem
    # quão bem o unistrut cruza a faixa ocupada pela viga, ou vice-versa.
#=====================================================================================================================

def get_principal_direction_xy_from_points(points_xyz):
    pts = [(x, y) for x, y, _ in points_xyz]
    if len(pts) < 2:
        return None

    cx = sum(x for x, _ in pts) / len(pts)
    cy = sum(y for _, y in pts) / len(pts)

    sxx = syy = sxy = 0.0
    for x, y in pts:
        dx = x - cx
        dy = y - cy
        sxx += dx * dx
        syy += dy * dy
        sxy += dx * dy

    trace = sxx + syy
    det = sxx * syy - sxy * sxy
    disc = max(trace * trace / 4 - det, 0.0)
    eig1 = trace / 2 + math.sqrt(disc)

    if abs(sxy) > 1e-9:
        vx = eig1 - syy
        vy = sxy
    else:
        if sxx >= syy:
            vx, vy = 1.0, 0.0
        else:
            vx, vy = 0.0, 1.0

    norm = math.sqrt(vx * vx + vy * vy)
    if norm == 0:
        return None

    return (vx / norm, vy / norm)
    # Calcula a direção principal do elemento no plano XY com base nos pontos geométricos.
    # Essa função é mais refinada do que simplesmente olhar a bounding box.
#=====================================================================================================================

def project_points_on_axis_xy(points_xyz, axis_vec):
    vx, vy = axis_vec
    return [x * vx + y * vy for x, y, _ in points_xyz]
    # Projeta os pontos do elemento sobre um eixo específico no plano XY. Dessa forma, 
    # é possível medir a extensão projetada do elemento, além das sobreposições entre
    # elementos (no eixo principal e no eixo perpendicular)
#=====================================================================================================================

def build_element_record(element):
    pts = get_shape_vertices_xyz(element)
    if not pts:
        return None

    bbox = get_bbox_and_center_from_points(pts)
    axis = get_main_axis_from_bbox(bbox)
    direction_xy = get_principal_direction_xy_from_points(pts)

    proj_main = None
    proj_perp = None
    if direction_xy is not None:
        perp = (-direction_xy[1], direction_xy[0])
        proj_main = project_points_on_axis_xy(pts, direction_xy)
        proj_perp = project_points_on_axis_xy(pts, perp)

    return {
        "entity": element,
        "points_xyz": pts,
        "bbox": bbox,
        "bbox_size": bbox_sizes(bbox),
        "summary": summarize_entity(element),
        "main_axis": axis,
        "direction_xy": direction_xy,
        "proj_main": proj_main,
        "proj_perp": proj_perp,
    }
    # Monta um registro geométrico completo do elemento. Essa função agrupa, para cada
    # viga ou unistrut, criando o objeto intermediário que basicamente será usado em todo o
    # restante da análise. É uma etapa de consolidação do pré-processamento geométrico.
#=====================================================================================================================

def get_projected_cross_metrics_from_records(uni_rec, beam_rec):
    uni_pts = uni_rec["points_xyz"]
    beam_pts = beam_rec["points_xyz"]
    uni_dir = uni_rec["direction_xy"]

    if not uni_pts or not beam_pts or uni_dir is None:
        return None

    uni_perp = (-uni_dir[1], uni_dir[0])

    uni_proj_main = uni_rec["proj_main"]
    uni_proj_perp = uni_rec["proj_perp"]
    beam_proj_main = project_points_on_axis_xy(beam_pts, uni_dir)
    beam_proj_perp = project_points_on_axis_xy(beam_pts, uni_perp)

    uni_main_min, uni_main_max = min(uni_proj_main), max(uni_proj_main)
    beam_main_min, beam_main_max = min(beam_proj_main), max(beam_proj_main)

    uni_perp_min, uni_perp_max = min(uni_proj_perp), max(uni_proj_perp)
    beam_perp_min, beam_perp_max = min(beam_proj_perp), max(beam_proj_perp)

    overlap_main = interval_overlap_length(uni_main_min, uni_main_max, beam_main_min, beam_main_max)
    overlap_perp = interval_overlap_length(uni_perp_min, uni_perp_max, beam_perp_min, beam_perp_max)

    uni_main_center = (uni_main_min + uni_main_max) / 2
    beam_main_center = (beam_main_min + beam_main_max) / 2
    main_center_offset = abs(uni_main_center - beam_main_center)

    return {
        "projected_overlap_main_mm": round(overlap_main, 2),
        "projected_overlap_perp_mm": round(overlap_perp, 2),
        "projected_main_center_offset_mm": round(main_center_offset, 2),
    }
    # Calcula métricas de relação geométrica projetada entre um unistrut e uma viga. Necessária
    # porque mesmo quando existe proximidade em bbox, pode haver divergência sob a lógica de
    # projeção dos elementos.
#=====================================================================================================================

def pair_by_projection_from_records(
    uni_rec,
    beam_rec,
    z_tol_mm: float,
    plan_tol_mm: float,
    min_parallel_overlap_mm: float,
    cross_tol_mm: float = 100.0
):
    uni_bbox = uni_rec["bbox"]
    beam_bbox = beam_rec["bbox"]
    uni_axis = uni_rec["main_axis"]
    beam_axis = beam_rec["main_axis"]

    z_gap = axis_gap(uni_bbox["zmin"], uni_bbox["zmax"], beam_bbox["zmin"], beam_bbox["zmax"])
    if z_gap > z_tol_mm:
        return None

    orientation = classify_relative_orientation(uni_axis, beam_axis)

    if orientation == "PERPENDICULAR":
        overlap_x = interval_overlap_length(
            uni_bbox["xmin"], uni_bbox["xmax"],
            beam_bbox["xmin"], beam_bbox["xmax"]
        )
        overlap_y = interval_overlap_length(
            uni_bbox["ymin"], uni_bbox["ymax"],
            beam_bbox["ymin"], beam_bbox["ymax"]
        )

        top_support_gap = abs(uni_bbox["zmin"] - beam_bbox["zmax"])
        bottom_support_gap = abs(uni_bbox["zmax"] - beam_bbox["zmin"])
        support_gap = min(top_support_gap, bottom_support_gap)

        if uni_axis == "X" and beam_axis == "Y":
            cross_miss_x = point_to_interval_gap(beam_bbox["cx"], uni_bbox["xmin"], uni_bbox["xmax"])
            cross_miss_y = point_to_interval_gap(uni_bbox["cy"], beam_bbox["ymin"], beam_bbox["ymax"])
        elif uni_axis == "Y" and beam_axis == "X":
            cross_miss_x = point_to_interval_gap(uni_bbox["cx"], beam_bbox["xmin"], beam_bbox["xmax"])
            cross_miss_y = point_to_interval_gap(beam_bbox["cy"], uni_bbox["ymin"], uni_bbox["ymax"])
        else:
            return None

        projected = get_projected_cross_metrics_from_records(uni_rec, beam_rec)
        if projected is None:
            return None

        overlap_main_mm = projected["projected_overlap_main_mm"]
        overlap_perp_mm = projected["projected_overlap_perp_mm"]
        main_center_offset_mm = projected["projected_main_center_offset_mm"]

        if (
            overlap_x > 0 and
            overlap_y > 0 and
            support_gap <= z_tol_mm and
            cross_miss_x <= cross_tol_mm and
            cross_miss_y <= cross_tol_mm and
            overlap_main_mm > 0
        ):
            return {
                "orientation": orientation,
                "z_gap_mm": round(z_gap, 2),
                "plan_miss_mm": round(math.sqrt(cross_miss_x**2 + cross_miss_y**2), 2),
                "x_miss_mm": round(cross_miss_x, 2),
                "y_miss_mm": round(cross_miss_y, 2),
                "main_overlap_mm": None,
                "overlap_x_mm": round(overlap_x, 2),
                "overlap_y_mm": round(overlap_y, 2),
                "top_support_gap_mm": round(top_support_gap, 2),
                "bottom_support_gap_mm": round(bottom_support_gap, 2),
                "support_position": "TOP" if top_support_gap <= bottom_support_gap else "BOTTOM",
                "projected_overlap_main_mm": overlap_main_mm,
                "projected_overlap_perp_mm": overlap_perp_mm,
                "projected_main_center_offset_mm": main_center_offset_mm,
            }

        return None

    if uni_axis == "X" and beam_axis == "X":
        main_overlap = interval_overlap_length(
            uni_bbox["xmin"], uni_bbox["xmax"],
            beam_bbox["xmin"], beam_bbox["xmax"]
        )
        side_gap = axis_gap(
            uni_bbox["ymin"], uni_bbox["ymax"],
            beam_bbox["ymin"], beam_bbox["ymax"]
        )

    elif uni_axis == "Y" and beam_axis == "Y":
        main_overlap = interval_overlap_length(
            uni_bbox["ymin"], uni_bbox["ymax"],
            beam_bbox["ymin"], beam_bbox["ymax"]
        )
        side_gap = axis_gap(
            uni_bbox["xmin"], uni_bbox["xmax"],
            beam_bbox["xmin"], beam_bbox["xmax"]
        )
    else:
        return None

    if side_gap <= plan_tol_mm and main_overlap >= min_parallel_overlap_mm:
        return {
            "orientation": orientation,
            "z_gap_mm": round(z_gap, 2),
            "plan_miss_mm": round(side_gap, 2),
            "x_miss_mm": None,
            "y_miss_mm": None,
            "main_overlap_mm": round(main_overlap, 2),
            "overlap_x_mm": None,
            "overlap_y_mm": None,
            "top_support_gap_mm": None,
            "bottom_support_gap_mm": None,
            "support_position": None,
            "projected_overlap_main_mm": None,
            "projected_overlap_perp_mm": None,
            "projected_main_center_offset_mm": None,
        }

    return None
    # Esta é a função central de decisão de pareamento entre unistrut e viga. Ela recebe
    # os registros geométricos dos dois elementos e testa se a relação atende aos critérios
    # definidos. Resumindo, ela responde: "Esses dois elementos podem ser considerados
    # um par unistrut-viga válido?" 
#=====================================================================================================================

def find_unistrut_beam_pairs_by_projection(
    model,
    z_tol_mm=5.0,
    plan_tol_mm=5.0,
    min_parallel_overlap_mm=100.0,
    cross_tol_mm=100.0
):
    t0 = time.time()

    beams = model.by_type("IfcBeam")
    unistruts = get_unistrut_entities(model)

    print(f"Beams encontrados: {len(beams)}")
    print(f"Unistruts encontrados: {len(unistruts)}")

    beam_data = []
    for i, beam in enumerate(beams):
        rec = build_element_record(beam)
        if rec:
            beam_data.append(rec)
        if (i + 1) % 10 == 0:
            print(f"Beam {i+1}/{len(beams)} processado | tempo: {round(time.time()-t0, 1)} s")

    print(f"Tempo após beam_data: {round(time.time()-t0, 1)} s")

    unistrut_data = []
    for i, uni in enumerate(unistruts):
        rec = build_element_record(uni)
        if rec:
            unistrut_data.append(rec)
        if (i + 1) % 20 == 0:
            print(f"Unistrut {i+1}/{len(unistruts)} processado | tempo: {round(time.time()-t0, 1)} s")

    print(f"Tempo após unistrut_data: {round(time.time()-t0, 1)} s")
    print(f"Beams com geometria válida: {len(beam_data)}")
    print(f"Unistruts com geometria válida: {len(unistrut_data)}")

    results = []

    for i, uni in enumerate(unistrut_data):
        if i % 10 == 0:
            print(f"Pareando unistrut {i+1}/{len(unistrut_data)} | tempo: {round(time.time()-t0, 1)} s")

        for beam in beam_data:
            pair_info = pair_by_projection_from_records(
                uni,
                beam,
                z_tol_mm=z_tol_mm,
                plan_tol_mm=plan_tol_mm,
                min_parallel_overlap_mm=min_parallel_overlap_mm,
                cross_tol_mm=cross_tol_mm
            )

            if pair_info is None:
                continue

            results.append({
                "unistrut_step_id": uni["summary"]["step_id"],
                "unistrut_global_id": uni["summary"]["global_id"],
                "unistrut_name": uni["summary"]["name"],
                "unistrut_tag": uni["summary"]["tag"],
                "unistrut_sts_name": uni["summary"]["sts_name"],
                "unistrut_axis": uni["main_axis"],

                "beam_step_id": beam["summary"]["step_id"],
                "beam_global_id": beam["summary"]["global_id"],
                "beam_name": beam["summary"]["name"],
                "beam_tag": beam["summary"]["tag"],
                "beam_axis": beam["main_axis"],

                "orientation": pair_info["orientation"],
                "z_gap_mm": pair_info["z_gap_mm"],
                "plan_miss_mm": pair_info["plan_miss_mm"],
                "x_miss_mm": pair_info["x_miss_mm"],
                "y_miss_mm": pair_info["y_miss_mm"],
                "main_overlap_mm": pair_info["main_overlap_mm"],

                "overlap_x_mm": pair_info.get("overlap_x_mm"),
                "overlap_y_mm": pair_info.get("overlap_y_mm"),
                "top_support_gap_mm": pair_info.get("top_support_gap_mm"),
                "bottom_support_gap_mm": pair_info.get("bottom_support_gap_mm"),
                "support_position": pair_info.get("support_position"),

                "projected_overlap_main_mm": pair_info.get("projected_overlap_main_mm"),
                "projected_overlap_perp_mm": pair_info.get("projected_overlap_perp_mm"),
                "projected_main_center_offset_mm": pair_info.get("projected_main_center_offset_mm"),

                "unistrut_bbox_sx_mm": round(uni["bbox_size"]["sx"], 2),
                "unistrut_bbox_sy_mm": round(uni["bbox_size"]["sy"], 2),
                "unistrut_bbox_sz_mm": round(uni["bbox_size"]["sz"], 2),

                "beam_bbox_sx_mm": round(beam["bbox_size"]["sx"], 2),
                "beam_bbox_sy_mm": round(beam["bbox_size"]["sy"], 2),
                "beam_bbox_sz_mm": round(beam["bbox_size"]["sz"], 2),
            })

    print(f"Tempo total: {round(time.time()-t0, 1)} s")
    return results
    # Aqui são armazenadas as informações de saída das funções anteriores. Basicamente,
    # ela executa o processo completo de busca de pares entre todos os unistruts e todas as
    # vigas do modelo. Ela:
    # 1. Coleta vigas do modelo;
    # 2. Coleta unistruts do modelo.
    # 3. Monta registro geométricos para todos
    # 4. Compara cada unistrut com cada viga
    # 5. Registra os pares unistrut-viga válidos
    # 6. retorna a lista final de resultados
#=====================================================================================================================

model = ifcopenshell.open("MAIC_V4.ifc")

all_pairs = find_unistrut_beam_pairs_by_projection(
    model,
    z_tol_mm=5.0,
    plan_tol_mm=5.0,
    min_parallel_overlap_mm=100.0,
    cross_tol_mm=100.0
)
    # Chamada da função principal

# print(f"\nTotal de pares unistrut-viga encontrados: {len(all_pairs)}")
# for row in all_pairs[:30]:
#     print(row)

Beams encontrados: 78
Unistruts encontrados: 281
Beam 10/78 processado | tempo: 5.1 s
Beam 20/78 processado | tempo: 10.5 s
Beam 30/78 processado | tempo: 12.9 s
Beam 40/78 processado | tempo: 15.1 s
Beam 50/78 processado | tempo: 15.5 s
Beam 60/78 processado | tempo: 18.4 s
Beam 70/78 processado | tempo: 19.2 s
Tempo após beam_data: 19.4 s
Unistrut 20/281 processado | tempo: 19.5 s
Unistrut 40/281 processado | tempo: 19.7 s
Unistrut 60/281 processado | tempo: 19.9 s
Unistrut 80/281 processado | tempo: 20.1 s
Unistrut 100/281 processado | tempo: 20.2 s
Unistrut 120/281 processado | tempo: 20.4 s
Unistrut 140/281 processado | tempo: 20.6 s
Unistrut 160/281 processado | tempo: 20.7 s
Unistrut 180/281 processado | tempo: 20.9 s
Unistrut 200/281 processado | tempo: 21.0 s
Unistrut 220/281 processado | tempo: 21.2 s
Unistrut 240/281 processado | tempo: 21.3 s
Unistrut 260/281 processado | tempo: 21.5 s
Unistrut 280/281 processado | tempo: 21.7 s
Tempo após unistrut_data: 21.7 s
Beams com ge

In [31]:
#4. RESUMO DAS ORIENTAÇÕES

orientation_counter = Counter(row["orientation"] for row in all_pairs)

print("Resumo das orientações:")
print(orientation_counter)

perpendicular_pairs = [r for r in all_pairs if r["orientation"] == "PERPENDICULAR"]
parallel_pairs = [r for r in all_pairs if r["orientation"] == "PARALLEL"]

Resumo das orientações:
Counter({'PERPENDICULAR': 35, 'PARALLEL': 23})


In [32]:
#5. REORGANIZAÇÃO DOS PARES PERPENDICULARES FAZENDO AGRUPAMENTO POR UNISTRUT
unistrut_to_beams = defaultdict(list)

for row in perpendicular_pairs:
    unistrut_to_beams[row["unistrut_tag"]].append({
        "beam_tag": row["beam_tag"],
        "beam_name": row["beam_name"],
        "orientation": row["orientation"],
        "z_gap_mm": row["z_gap_mm"],
        "plan_miss_mm": row["plan_miss_mm"],
    })

# Este trecho reorganiza os pares geométricos classificados como perpendiculares em uma 
# estrutura agrupada por unistrut. Para cada suporte identificado, são armazenadas as 
# vigas associadas e algumas métricas relevantes da relação geométrica, como afastamento 
# vertical (z_gap_mm) e desalinhamento em planta (plan_miss_mm). Essa organização facilita 
# etapas posteriores de análise, seleção de candidatos e geração de relatórios.

In [33]:
#7. RESUMO DE VIGAS POR ORIENTAÇÃO RELATIVA AO UNISTRUT
pair_count_all = Counter(row["unistrut_tag"] for row in all_pairs)
orientation_count = Counter(row["orientation"] for row in all_pairs)

print("Resumo das orientações:")
print(orientation_count)

distribution = Counter(pair_count_all.values())
print("\nDistribuição de nº de vigas por unistrut:")
for n, qty in sorted(distribution.items()):
    print(f"{n} viga(s): {qty} unistrut(s)")

Resumo das orientações:
Counter({'PERPENDICULAR': 35, 'PARALLEL': 23})

Distribuição de nº de vigas por unistrut:
1 viga(s): 28 unistrut(s)
2 viga(s): 11 unistrut(s)
3 viga(s): 1 unistrut(s)
5 viga(s): 1 unistrut(s)


In [34]:
#8. RESUMO DE UNISTRUTS

all_unistrut_tags = set()

for e in get_unistrut_entities(model):
    tag = get_attr(e, "Tag")
    if tag:
        all_unistrut_tags.add(tag)

paired_unistrut_tags = set(row["unistrut_tag"] for row in all_pairs)

missing = sorted(all_unistrut_tags - paired_unistrut_tags)

print(f"Total de unistruts no modelo: {len(all_unistrut_tags)}")
print(f"Total de unistruts com pelo menos 1 viga associada: {len(paired_unistrut_tags)}")
print(f"Total de unistruts sem viga associada: {len(missing)}")
print("\nAmostra de unistruts sem par:")
for tag in missing[:30]:
    print(tag)

Total de unistruts no modelo: 281
Total de unistruts com pelo menos 1 viga associada: 41
Total de unistruts sem viga associada: 240

Amostra de unistruts sem par:
10001338
10001342
10001343
10001354
10001357
10001358
10011611
10011612
10011613
10011658
10050256
10091775
10095928
10097953
10098185
10098187
10098188
10098189
10098191
10098192
10099588
10099589
10099590
10104425
10104577
10123698
10123702
10161244
10166518
10166830


In [35]:
#10 - CHECAGEM DE OVERHANG

def get_entity_by_tag(model, ifc_class: str, tag_value: str):
    for e in model.by_type(ifc_class):
        if get_attr(e, "Tag") == str(tag_value):
            return e
    return None

def compute_perpendicular_overhangs(row, model):
    """
    Calcula os overhangs esquerdo e direito do Unistrut em relação à viga
    para pares PERPENDICULAR.
    """
    if row["orientation"] != "PERPENDICULAR":
        return None

    uni = get_entity_by_tag(model, "IfcBuildingElementProxy", str(row["unistrut_tag"]))
    beam = get_entity_by_tag(model, "IfcBeam", str(row["beam_tag"]))

    if uni is None or beam is None:
        return None

    uni_bbox = get_bbox_and_center(uni)
    beam_bbox = get_bbox_and_center(beam)

    if not uni_bbox or not beam_bbox:
        return None

    uni_axis = row["unistrut_axis"]
    beam_axis = row["beam_axis"]

    if uni_axis == "X" and beam_axis == "Y":
        left_overhang = beam_bbox["xmin"] - uni_bbox["xmin"]
        right_overhang = uni_bbox["xmax"] - beam_bbox["xmax"]

        left_has_beam = beam_bbox["cx"] < uni_bbox["cx"]
        right_has_beam = beam_bbox["cx"] > uni_bbox["cx"]

    elif uni_axis == "Y" and beam_axis == "X":
        left_overhang = beam_bbox["ymin"] - uni_bbox["ymin"]
        right_overhang = uni_bbox["ymax"] - beam_bbox["ymax"]

        left_has_beam = beam_bbox["cy"] < uni_bbox["cy"]
        right_has_beam = beam_bbox["cy"] > uni_bbox["cy"]

    else:
        return None

    return {
        "left_overhang_mm": round(left_overhang, 2),
        "right_overhang_mm": round(right_overhang, 2),
        "min_overhang_mm": round(min(left_overhang, right_overhang), 2),
        "max_overhang_mm": round(max(left_overhang, right_overhang), 2),
        "has_overhang_both_sides": (left_overhang > 0 and right_overhang > 0),
        "left_has_beam_this_pair": left_has_beam,
        "right_has_beam_this_pair": right_has_beam,
    }


def evaluate_revised_overhang_rule(perpendicular_pairs, model):
    """
    Avalia a nova regra:
    1) Se não houver overhang dos dois lados:
       - PASS se houver vigas em ambos os lados do unistrut
       - FAIL caso contrário
    2) Se houver overhang dos dois lados:
       - pega o menor
       - PASS se 50 <= menor <= 90
       - FAIL caso contrário
    """
    enriched_rows = []

    for row in perpendicular_pairs:
        info = compute_perpendicular_overhangs(row, model)
        if info is None:
            continue

        new_row = row.copy()
        new_row.update(info)
        enriched_rows.append(new_row)

    # agrupa por unistrut para saber se existem vigas em ambos os lados
    from collections import defaultdict
    grouped = defaultdict(list)

    for row in enriched_rows:
        grouped[row["unistrut_tag"]].append(row)

    final_results = []

    for unistrut_tag, rows in grouped.items():
        # verifica existência de vigas à esquerda e à direita considerando todos os pares perpendiculares
        has_left_beam_any = any(r["left_has_beam_this_pair"] for r in rows)
        has_right_beam_any = any(r["right_has_beam_this_pair"] for r in rows)

        for row in rows:
            has_overhang_both_sides = row["has_overhang_both_sides"]

            if not has_overhang_both_sides:
                rule_pass = has_left_beam_any and has_right_beam_any
                rule_reason = (
                    "NO_OVERHANG_BUT_BEAMS_BOTH_SIDES_PASS"
                    if rule_pass else
                    "NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL"
                )
            else:
                min_ov = row["min_overhang_mm"]
                rule_pass = (50.0 <= min_ov <= 90.0)
                rule_reason = (
                    "OVERHANG_MIN_BETWEEN_50_AND_90_PASS"
                    if rule_pass else
                    "OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL"
                )

            final_row = row.copy()
            final_row["has_left_beam_any"] = has_left_beam_any
            final_row["has_right_beam_any"] = has_right_beam_any
            final_row["rule_pass"] = rule_pass
            final_row["rule_reason"] = rule_reason

            final_results.append(final_row)

    return final_results

In [36]:
# 11. REGRA FINAL DE PASS / FAIL

def compute_projected_overhangs(row, model):
    """
    Calcula overhang do Unistrut em relação à viga por projeção
    no eixo principal real do Unistrut.
    """
    if row["orientation"] != "PERPENDICULAR":
        return None

    uni = get_entity_by_tag(model, "IfcBuildingElementProxy", str(row["unistrut_tag"]))
    beam = get_entity_by_tag(model, "IfcBeam", str(row["beam_tag"]))

    if uni is None or beam is None:
        return None

    uni_rec = build_element_record(uni)
    beam_rec = build_element_record(beam)

    if uni_rec is None or beam_rec is None:
        return None

    uni_pts = uni_rec["points_xyz"]
    beam_pts = beam_rec["points_xyz"]
    uni_dir = uni_rec["direction_xy"]

    if not uni_pts or not beam_pts or uni_dir is None:
        return None

    uni_proj = uni_rec["proj_main"]
    beam_proj = project_points_on_axis_xy(beam_pts, uni_dir)

    uni_min = min(uni_proj)
    uni_max = max(uni_proj)
    beam_min = min(beam_proj)
    beam_max = max(beam_proj)

    left_overhang = beam_min - uni_min
    right_overhang = uni_max - beam_max

    return {
        "projected_unistrut_length_mm": round(uni_max - uni_min, 2),
        "left_overhang_mm": round(left_overhang, 2),
        "right_overhang_mm": round(right_overhang, 2),
        "min_overhang_mm": round(min(left_overhang, right_overhang), 2),
        "max_overhang_mm": round(max(left_overhang, right_overhang), 2),
        "has_overhang_both_sides": (left_overhang > 0 and right_overhang > 0),
    }


def evaluate_pass_fail(perpendicular_pairs, model):
    """
    Regra:
    1. Se NÃO houver overhang nos dois lados:
       - PASS se houver vigas em ambos os lados do Unistrut
       - FAIL caso contrário
    2. Se houver overhang nos dois lados:
       - PASS se o menor overhang estiver entre 50 e 90 mm
       - FAIL caso contrário
    """
    from collections import defaultdict

    enriched_rows = []

    for row in perpendicular_pairs:
        info = compute_projected_overhangs(row, model)
        if info is None:
            continue

        new_row = row.copy()
        new_row.update(info)
        enriched_rows.append(new_row)

    grouped = defaultdict(list)
    for row in enriched_rows:
        grouped[row["unistrut_tag"]].append(row)

    final_results = []

    for unistrut_tag, rows in grouped.items():
        has_left_beam_any = False
        has_right_beam_any = False

        for row in rows:
            uni = get_entity_by_tag(model, "IfcBuildingElementProxy", str(row["unistrut_tag"]))
            beam = get_entity_by_tag(model, "IfcBeam", str(row["beam_tag"]))

            uni_rec = build_element_record(uni)
            beam_rec = build_element_record(beam)

            if uni_rec is None or beam_rec is None:
                continue

            uni_proj = uni_rec["proj_main"]
            beam_proj = project_points_on_axis_xy(beam_rec["points_xyz"], uni_rec["direction_xy"])

            uni_center = (min(uni_proj) + max(uni_proj)) / 2
            beam_center = (min(beam_proj) + max(beam_proj)) / 2

            if beam_center < uni_center:
                has_left_beam_any = True
            if beam_center > uni_center:
                has_right_beam_any = True

        for row in rows:
            if not row["has_overhang_both_sides"]:
                rule_pass = has_left_beam_any and has_right_beam_any
                rule_reason = (
                    "NO_OVERHANG_BUT_BEAMS_BOTH_SIDES_PASS"
                    if rule_pass else
                    "NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL"
                )
            else:
                min_ov = row["min_overhang_mm"]
                rule_pass = (50.0 <= min_ov <= 90.0)
                rule_reason = (
                    "OVERHANG_MIN_BETWEEN_50_AND_90_PASS"
                    if rule_pass else
                    "OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL"
                )

            final_row = row.copy()
            final_row["has_left_beam_any"] = has_left_beam_any
            final_row["has_right_beam_any"] = has_right_beam_any
            final_row["rule_pass"] = rule_pass
            final_row["rule_reason"] = rule_reason

            final_results.append(final_row)

    return final_results
rule_results = evaluate_pass_fail(perpendicular_pairs, model)



Erro ao obter vértices XYZ do elemento 72495 - STS_StructuralFraming_Single_P1000:P1000 (Single unistrut):10177171: <built-in function create_shape> returned a result with an exception set


In [ ]:
# 12. REPORT DE FAILS E PASSES
fails = [r for r in rule_results if not r["rule_pass"]]
print(f"Total de FAILs: {len(fails)}")
for row in fails[:30]:
    print({
        "unistrut_tag": row["unistrut_tag"],
        "beam_tag": row["beam_tag"],
        "left_overhang_mm": row["left_overhang_mm"],
        "right_overhang_mm": row["right_overhang_mm"],
        "min_overhang_mm": row["min_overhang_mm"],
        "rule_reason": row["rule_reason"],
    })

    fails = [r for r in rule_results if not r["rule_pass"]]

passes = [r for r in rule_results if r["rule_pass"]]
print(f"Total de PASSes: {len(passes)}")
for row in passes[:30]:
    print({
        "unistrut_tag": row["unistrut_tag"],
        "beam_tag": row["beam_tag"],
        "left_overhang_mm": row["left_overhang_mm"],
        "right_overhang_mm": row["right_overhang_mm"],
        "min_overhang_mm": row["min_overhang_mm"],
        "rule_reason": row["rule_reason"],
    })

Total de FAILs: 29
{'unistrut_tag': '9005875', 'beam_tag': '4503280', 'left_overhang_mm': 0.0, 'right_overhang_mm': 2789.93, 'min_overhang_mm': 0.0, 'rule_reason': 'OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL'}
{'unistrut_tag': '9094410', 'beam_tag': '3550994', 'left_overhang_mm': 954.42, 'right_overhang_mm': -154.43, 'min_overhang_mm': -154.43, 'rule_reason': 'NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL'}
{'unistrut_tag': '9095030', 'beam_tag': '3550994', 'left_overhang_mm': 954.42, 'right_overhang_mm': -150.08, 'min_overhang_mm': -150.08, 'rule_reason': 'NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL'}
{'unistrut_tag': '9097052', 'beam_tag': '3554366', 'left_overhang_mm': 2115.3, 'right_overhang_mm': 3874.95, 'min_overhang_mm': 2115.3, 'rule_reason': 'OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL'}
{'unistrut_tag': '9097344', 'beam_tag': '4503280', 'left_overhang_mm': 954.42, 'right_overhang_mm': -150.01, 'min_overhang_mm': -150.01, 'rule_reason': 'NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL'}
{'unistrut_tag': '

In [ ]:
# 13. COMPILAÇÃO DE DATABASE PARA GERAÇÃO DE RELATÓRIO

def build_checklist_dataframe(results):
    rows = []

    for r in results:
        rows.append({
            "Unistrut_Tag": r.get("unistrut_tag"),
            "Beam_Tag": r.get("beam_tag"),
            "Orientation": r.get("orientation"),
            "Projected_Unistrut_Length_mm": r.get("projected_unistrut_length_mm"),
            "Left_Overhang_mm": r.get("left_overhang_mm"),
            "Right_Overhang_mm": r.get("right_overhang_mm"),
            "Min_Overhang_mm": r.get("min_overhang_mm"),
            "Max_Overhang_mm": r.get("max_overhang_mm"),
            "Has_Overhang_Both_Sides": r.get("has_overhang_both_sides"),
            "Has_Left_Beam_Any": r.get("has_left_beam_any"),
            "Has_Right_Beam_Any": r.get("has_right_beam_any"),
            "Rule_Pass": r.get("rule_pass"),
            "Rule_Reason": r.get("rule_reason"),
            "z_gap_mm": r.get("z_gap_mm"),
            "plan_miss_mm": r.get("plan_miss_mm"),
            "projected_overlap_main_mm": r.get("projected_overlap_main_mm"),
            "projected_overlap_perp_mm": r.get("projected_overlap_perp_mm"),
            "projected_main_center_offset_mm": r.get("projected_main_center_offset_mm"),
        })

    return pd.DataFrame(rows)


df_checklist = build_checklist_dataframe(rule_results)

print("Prévia do checklist:")
print(df_checklist.head(10))
print("\nTotal de linhas:", len(df_checklist))

Prévia do checklist:
  Unistrut_Tag Beam_Tag    Orientation  Projected_Unistrut_Length_mm  \
0      8995328  4460405  PERPENDICULAR                        900.00   
1      9005875  4503280  PERPENDICULAR                       2989.93   
2      9094410  3550994  PERPENDICULAR                        999.99   
3      9095030  3550994  PERPENDICULAR                       1004.34   
4      9097052  3554355  PERPENDICULAR                       6190.24   
5      9097052  3554366  PERPENDICULAR                       6190.24   
6      9097344  4503280  PERPENDICULAR                       1004.42   
7      9097403  4503280  PERPENDICULAR                       1400.00   
8      9098632  4503280  PERPENDICULAR                        999.99   
9      9509116  4593157  PERPENDICULAR                       3996.00   

   Left_Overhang_mm  Right_Overhang_mm  Min_Overhang_mm  Max_Overhang_mm  \
0            650.00              50.00            50.00           650.00   
1              0.00            278

In [ ]:
#14. RESUMO EXECUTIVO DOS RESULTADOS

total_rows = len(df_checklist)
pass_count = int(df_checklist["Rule_Pass"].fillna(False).sum())
fail_count = total_rows - pass_count

reason_counter = Counter(df_checklist["Rule_Reason"].fillna("NO_REASON"))

print("Resumo executivo")
print("-" * 40)
print(f"Total de verificações: {total_rows}")
print(f"PASS: {pass_count}")
print(f"FAIL: {fail_count}")

print("\nMotivos:")
for reason, qty in reason_counter.most_common():
    print(f"{reason}: {qty}")

Resumo executivo
----------------------------------------
Total de verificações: 35
PASS: 6
FAIL: 29

Motivos:
OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL: 22
NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL: 7
OVERHANG_MIN_BETWEEN_50_AND_90_PASS: 5
NO_OVERHANG_BUT_BEAMS_BOTH_SIDES_PASS: 1


In [ ]:
#15. EXPORTAÇÃO PARA EXCEL

output_excel = "unistrut_compliance_checklist.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    df_checklist.to_excel(writer, sheet_name="Checklist", index=False)

    summary_df = pd.DataFrame([
        {"Metric": "Total checks", "Value": total_rows},
        {"Metric": "PASS", "Value": pass_count},
        {"Metric": "FAIL", "Value": fail_count},
    ])
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"Arquivo gerado: {output_excel}")

Arquivo gerado: unistrut_compliance_checklist.xlsx


In [ ]:
# 16. GERAÇÃO DE RELATÓRIO EM HTML

from pathlib import Path
from datetime import datetime
from collections import Counter


def generate_html_report(
    ifc_path: str,
    df_checklist,
    pass_count: int,
    fail_count: int,
    reason_counter,
    all_pairs: list = None,
    output_path: str = None,
):
    ifc_name     = Path(ifc_path).stem
    total_checks = pass_count + fail_count
    pass_pct     = round(pass_count / total_checks * 100, 1) if total_checks else 0
    fail_pct     = round(fail_count / total_checks * 100, 1) if total_checks else 0
    total_pairs  = len(all_pairs) if all_pairs else 0
    generated_at = datetime.now().strftime("%d %b %Y  %H:%M")

    # orientation counts
    n_perp = n_para = 0
    if all_pairs:
        oc = Counter(r.get("orientation", "") for r in all_pairs)
        n_perp = oc.get("PERPENDICULAR", 0)
        n_para = oc.get("PARALLEL", 0)

    # reason breakdown
    reason_colors = {
        "OVERHANG_MIN_BETWEEN_50_AND_90_PASS":       ("#00e5a0", "PASS"),
        "NO_OVERHANG_BUT_BEAMS_BOTH_SIDES_PASS":     ("#00e5a0", "PASS"),
        "OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL":        ("#ff4d6d", "FAIL"),
        "NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL": ("#ff4d6d", "FAIL"),
    }
    total_reasons    = sum(reason_counter.values()) or 1
    reason_rows_html = ""
    for reason, qty in reason_counter.most_common():
        color, status = reason_colors.get(reason, ("#aaa", "—"))
        bar_w = round(qty / total_reasons * 100)
        label = reason.replace("_", " ").title()
        reason_rows_html += f"""
        <div class="reason-row">
          <div class="reason-meta">
            <span class="reason-badge" style="background:{color}22;color:{color}">{status}</span>
            <span class="reason-label">{label}</span>
            <span class="reason-count">{qty}</span>
          </div>
          <div class="reason-bar-bg">
            <div class="reason-bar-fill" style="width:{bar_w}%;background:{color}"></div>
          </div>
        </div>"""

    # all elements table rows (FAIL first, then PASS)
    df_sorted      = df_checklist.sort_values("Rule_Pass", ascending=True)
    all_rows_html  = ""
    for _, row in df_sorted.iterrows():
        is_pass      = bool(row.get("Rule_Pass", False))
        reason_raw   = str(row.get("Rule_Reason", ""))
        reason_short = reason_raw.replace("_", " ").replace(" FAIL", "").replace(" PASS", "").title()
        min_ov   = row.get("Min_Overhang_mm")
        left_ov  = row.get("Left_Overhang_mm")
        right_ov = row.get("Right_Overhang_mm")
        min_str  = f"{min_ov:.1f} mm" if min_ov  is not None else "—"
        l_str    = f"{left_ov:.1f}"  if left_ov is not None else "—"
        r_str    = f"{right_ov:.1f}" if right_ov is not None else "—"
        pill     = '<span class="pill pill-pass">PASS</span>' if is_pass else '<span class="pill pill-fail">FAIL</span>'
        all_rows_html += f"""
        <tr>
          <td class="mono">{row.get('Unistrut_Tag','—')}</td>
          <td class="mono">{row.get('Beam_Tag','—')}</td>
          <td>{row.get('Orientation','—')}</td>
          <td>{min_str}</td>
          <td class="overhang-cell">{l_str} / {r_str}</td>
          <td>{pill}</td>
          <td class="reason-text">{reason_short}</td>
        </tr>"""

    # donut — stroke-dasharray on a <circle> = true ring, no lines to centre
    CIRC      = 452.39          # 2 * pi * 72
    GAP       = 4               # small gap between arcs
    p_dash    = max(round(pass_pct / 100 * CIRC, 2) - GAP, 0)
    f_dash    = max(round(fail_pct / 100 * CIRC, 2) - GAP, 0)
    f_offset  = round(CIRC - round(pass_pct / 100 * CIRC, 2), 2)

    donut_svg = f"""
    <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg" class="donut-svg">
      <defs>
        <filter id="gp"><feGaussianBlur stdDeviation="4" result="b"/>
          <feMerge><feMergeNode in="b"/><feMergeNode in="SourceGraphic"/></feMerge></filter>
        <filter id="gf"><feGaussianBlur stdDeviation="4" result="b"/>
          <feMerge><feMergeNode in="b"/><feMergeNode in="SourceGraphic"/></feMerge></filter>
      </defs>
      <circle cx="100" cy="100" r="72" fill="none" stroke="#1e2530" stroke-width="22"/>
      <circle cx="100" cy="100" r="72" fill="none" stroke="#ff4d6d" stroke-width="22"
              stroke-dasharray="{f_dash} {CIRC}" stroke-dashoffset="{f_offset}"
              stroke-linecap="round" filter="url(#gf)"
              transform="rotate(-90 100 100)"/>
      <circle cx="100" cy="100" r="72" fill="none" stroke="#00e5a0" stroke-width="22"
              stroke-dasharray="{p_dash} {CIRC}" stroke-dashoffset="0"
              stroke-linecap="round" filter="url(#gp)"
              transform="rotate(-90 100 100)"/>
      <text x="100" y="93" text-anchor="middle" font-size="28" font-weight="700"
            fill="#f0f4ff" font-family="'DM Mono',monospace">{pass_pct}%</text>
      <text x="100" y="113" text-anchor="middle" font-size="10" fill="#6b7a99"
            font-family="'DM Sans',sans-serif" letter-spacing="2">COMPLIANT</text>
    </svg>"""

    # orientation bars
    orient_total = n_perp + n_para or 1
    perp_bar     = round(n_perp / orient_total * 100)
    para_bar     = round(n_para / orient_total * 100)
    orient_block = ""
    if all_pairs:
        orient_block = f"""
  <div class="panel orient-panel">
    <div class="panel-title">Pair Orientation Distribution</div>
    <div class="orient-row">
      <div class="orient-meta">
        <span class="orient-label">&#x22A5; Perpendicular</span>
        <span class="orient-count">{n_perp}</span>
      </div>
      <div class="reason-bar-bg">
        <div class="reason-bar-fill" style="width:{perp_bar}%;background:#4d9fff"></div>
      </div>
    </div>
    <div class="orient-row">
      <div class="orient-meta">
        <span class="orient-label">&#x2225; Parallel</span>
        <span class="orient-count">{n_para}</span>
      </div>
      <div class="reason-bar-bg">
        <div class="reason-bar-fill" style="width:{para_bar}%;background:#a78bfa"></div>
      </div>
    </div>
  </div>"""

    html = f"""<!DOCTYPE html>
<html lang="pt">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>IFC Report — {ifc_name}</title>
<link rel="preconnect" href="https://fonts.googleapis.com"/>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=DM+Sans:wght@300;400;500;600&family=Syne:wght@700;800&display=swap" rel="stylesheet"/>
<style>
  :root{{--bg:#0b0f1a;--surface:#111827;--surface2:#1a2235;--border:#1e2d45;--text:#e8eeff;--muted:#6b7a99;--pass:#00e5a0;--fail:#ff4d6d;--accent:#4d9fff;--radius:14px;}}
  *,*::before,*::after{{box-sizing:border-box;margin:0;padding:0;}}
  body{{background:var(--bg);color:var(--text);font-family:'DM Sans',sans-serif;font-size:15px;line-height:1.6;min-height:100vh;}}
  body::before{{content:'';position:fixed;inset:0;pointer-events:none;z-index:0;background-image:url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='0.04'/%3E%3C/svg%3E");opacity:.35;}}
  .wrap{{position:relative;z-index:1;max-width:1100px;margin:0 auto;padding:48px 24px 80px;}}
  .header{{margin-bottom:52px;animation:fadeUp .5s ease both;}}
  .badge-ifc{{display:inline-flex;align-items:center;gap:8px;background:var(--surface);border:1px solid var(--border);border-radius:100px;padding:6px 14px;font-family:'DM Mono',monospace;font-size:12px;color:var(--muted);letter-spacing:.5px;margin-bottom:18px;}}
  .badge-ifc::before{{content:'';display:inline-block;width:7px;height:7px;border-radius:50%;background:var(--accent);box-shadow:0 0 8px var(--accent);}}
  h1{{font-family:'Syne',sans-serif;font-size:clamp(32px,5vw,56px);font-weight:800;letter-spacing:-1.5px;line-height:1.05;background:linear-gradient(135deg,#e8eeff 0%,#6b9fff 100%);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text;}}
  .subtitle{{color:var(--muted);font-size:14px;margin-top:10px;font-weight:300;}}
  .meta{{font-family:'DM Mono',monospace;font-size:11px;color:var(--muted);margin-top:6px;letter-spacing:.5px;}}
  .cards{{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:16px;margin-bottom:40px;animation:fadeUp .5s .1s ease both;}}
  .card{{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:24px 24px 20px;position:relative;overflow:hidden;transition:border-color .2s,transform .2s;}}
  .card:hover{{border-color:#2a3d5e;transform:translateY(-2px);}}
  .card::after{{content:'';position:absolute;top:0;left:0;right:0;height:2px;background:var(--card-accent,var(--accent));}}
  .card-label{{font-size:11px;font-weight:600;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:10px;}}
  .card-value{{font-family:'Syne',sans-serif;font-size:42px;font-weight:700;line-height:1;}}
  .card-sub{{font-size:12px;color:var(--muted);margin-top:6px;}}
  .card-pass{{--card-accent:var(--pass);}} .card-fail{{--card-accent:var(--fail);}} .card-total{{--card-accent:var(--accent);}} .card-uni{{--card-accent:#f5a623;}}
  .two-col{{display:grid;grid-template-columns:1fr 1.7fr;gap:24px;margin-bottom:24px;align-items:start;animation:fadeUp .5s .2s ease both;}}
  @media(max-width:700px){{.two-col{{grid-template-columns:1fr;}}}}
  .panel{{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:28px;}}
  .panel-title{{font-size:11px;font-weight:600;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:20px;}}
  .donut-svg{{width:100%;max-width:175px;display:block;margin:0 auto 20px;}}
  .donut-legend{{display:flex;justify-content:center;gap:20px;}}
  .legend-item{{display:flex;align-items:center;gap:7px;font-size:13px;}}
  .legend-dot{{width:10px;height:10px;border-radius:50%;flex-shrink:0;}}
  .reason-row{{margin-bottom:18px;}} .reason-meta{{display:flex;align-items:center;gap:10px;margin-bottom:7px;}}
  .reason-badge{{font-size:10px;font-weight:700;letter-spacing:1.5px;padding:2px 8px;border-radius:100px;flex-shrink:0;}}
  .reason-label{{font-size:13px;color:var(--text);flex:1;}} .reason-count{{font-family:'DM Mono',monospace;font-size:13px;color:var(--muted);}}
  .reason-bar-bg{{background:var(--surface2);border-radius:100px;height:5px;overflow:hidden;}}
  .reason-bar-fill{{height:100%;border-radius:100px;}}
  .orient-panel{{animation:fadeUp .5s .25s ease both;margin-bottom:24px;}}
  .orient-row{{margin-bottom:14px;}} .orient-meta{{display:flex;align-items:center;gap:10px;margin-bottom:7px;}}
  .orient-label{{font-size:14px;color:var(--text);flex:1;font-weight:500;}}
  .orient-count{{font-family:'DM Mono',monospace;font-size:13px;color:var(--muted);}}
  .table-panel{{animation:fadeUp .5s .3s ease both;}}
  .table-wrap{{overflow-x:auto;}}
  table{{width:100%;border-collapse:collapse;font-size:13px;}}
  thead tr{{background:var(--surface2);}}
  th{{padding:12px 14px;text-align:left;font-size:10px;font-weight:600;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);border-bottom:1px solid var(--border);white-space:nowrap;}}
  td{{padding:11px 14px;border-bottom:1px solid var(--border);vertical-align:middle;}}
  tbody tr{{transition:background .15s;}} tbody tr:hover{{background:var(--surface2);}}
  .mono{{font-family:'DM Mono',monospace;font-size:12px;color:var(--muted);}}
  .pill{{display:inline-block;padding:2px 10px;border-radius:100px;font-size:10px;font-weight:700;letter-spacing:1px;}}
  .pill-pass{{background:#00e5a018;color:var(--pass);}} .pill-fail{{background:#ff4d6d18;color:var(--fail);}}
  .reason-text{{font-size:12px;color:var(--muted);max-width:260px;}}
  .overhang-cell{{font-family:'DM Mono',monospace;font-size:12px;}}
  .footer{{text-align:center;color:var(--muted);font-size:12px;margin-top:60px;padding-top:24px;border-top:1px solid var(--border);letter-spacing:.5px;}}
  @keyframes fadeUp{{from{{opacity:0;transform:translateY(18px);}}to{{opacity:1;transform:translateY(0);}}}}
</style>
</head>
<body>
<div class="wrap">
  <div class="header">
    <div class="badge-ifc">IFC · {ifc_name}</div>
    <h1>Compliance<br>Report</h1>
    <p class="subtitle">Unistrut&#x2013;Beam Pairing Analysis &middot; Overhang Rule Evaluation</p>
    <p class="meta">Generated {generated_at}</p>
  </div>
  <div class="cards">
    <div class="card card-uni">
      <div class="card-label">Total Pairs</div>
      <div class="card-value" style="color:#f5a623">{total_pairs}</div>
      <div class="card-sub">{n_perp} perpendicular · {n_para} parallel</div>
    </div>
    <div class="card card-total">
      <div class="card-label">Total Checks</div>
      <div class="card-value">{total_checks}</div>
      <div class="card-sub">Perpendicular pairs evaluated for overhang rule</div>
    </div>
    <div class="card card-pass">
      <div class="card-label">Pass</div>
      <div class="card-value" style="color:var(--pass)">{pass_count}</div>
      <div class="card-sub">{pass_pct}% of perpendicular pairs</div>
    </div>
    <div class="card card-fail">
      <div class="card-label">Fail</div>
      <div class="card-value" style="color:var(--fail)">{fail_count}</div>
      <div class="card-sub">{fail_pct}% of perpendicular pairs</div>
    </div>
  </div>
  <div class="two-col">
    <div class="panel">
      <div class="panel-title">Compliance Rate · Perpendicular Pairs</div>
      {donut_svg}
      <div class="donut-legend">
        <div class="legend-item"><div class="legend-dot" style="background:var(--pass)"></div><span>{pass_count} Pass</span></div>
        <div class="legend-item"><div class="legend-dot" style="background:var(--fail)"></div><span>{fail_count} Fail</span></div>
      </div>
    </div>
    <div class="panel">
      <div class="panel-title">Breakdown by Rule · Perpendicular Pairs</div>
      {reason_rows_html}
    </div>
  </div>
  {orient_block}
  <div class="panel table-panel">
    <div class="panel-title" style="margin-bottom:18px">Elements Summary</div>
    <div class="table-wrap">
      <table>
        <thead>
          <tr>
            <th>Unistrut Tag</th><th>Beam Tag</th><th>Orientation</th>
            <th>Min Overhang</th><th>Left / Right (mm)</th><th>Status</th><th>Reason</th>
          </tr>
        </thead>
        <tbody>
          {all_rows_html if all_rows_html else '<tr><td colspan="7" style="text-align:center;color:var(--muted);padding:32px">No results to display.</td></tr>'}
        </tbody>
      </table>
    </div>
  </div>
  <div class="footer">IFC Extractor &middot; {ifc_name} &middot; {generated_at}</div>
</div>
</body>
</html>"""

    if output_path is None:
        output_path = Path(ifc_path).with_suffix("").name + "_report.html"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Relatorio gerado: {output_path}")
    return output_path


# ── CHAMADA — descomenta e cola no fim do notebook ────────────────────────────

generate_html_report(
    ifc_path      = ifc_path,
    df_checklist  = df_checklist,
    pass_count    = pass_count,
    fail_count    = fail_count,
    reason_counter= reason_counter,
    all_pairs     = all_pairs,
)

Relatorio gerado: MAIC_ROOM_02_report.html


'MAIC_ROOM_02_report.html'

In [ ]:
# 17 — INSTALAÇÃO E TESTE DE CONEXÃO COM A ANTHROPIC

import anthropic

def test_anthropic_connection() -> bool:
    try:
        client = anthropic.Anthropic()                  
        msg = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=64,
            messages=[{"role": "user", "content": "Responda apenas: CONEXÃO OK"}],
        )
        reply = msg.content[0].text.strip()
        print(f"✅  Anthropic conectada. Resposta: '{reply}'")
        return True
    except anthropic.AuthenticationError:
        print("❌  Falha de autenticação — verifique ANTHROPIC_API_KEY.")
        return False
    except Exception as e:
        print(f"❌  Erro inesperado: {e}")
        return False

#   Faz uma chamada mínima à API da Anthropic para confirmar que
#   a chave está válida e o endpoint responde.
#   Retorna True se a conexão for bem-sucedida.

test_anthropic_connection()


✅  Anthropic conectada. Resposta: 'CONEXÃO OK'


True

In [ ]:


# =============================================================================
# 18 — TOOL 01 · IFC Compliance Checker
# =============================================================================
# Esta tool executa toda a pipeline de análise IFC já definida nas células
# anteriores do notebook (find_unistrut_beam_pairs_by_projection,
# evaluate_pass_fail, build_checklist_dataframe, etc.) e devolve um
# resumo estruturado em texto para o agente.
# =============================================================================

from crewai.tools import tool

@tool("ifc_compliance_checker")
def ifc_compliance_checker(ifc_file_path: str) -> str:
    """
    Executa o pipeline completo de verificação de compliance de suportes
    em vigas a partir de um arquivo IFC.

    Parâmetros
    ----------
    ifc_file_path : str
        Caminho absoluto ou relativo para o arquivo .ifc a ser analisado.

    Retorno
    -------
    str
        Resumo em texto com:
        - totais de elementos encontrados
        - pares unistrut-viga (paralelo / perpendicular)
        - contagem de PASS e FAIL
        - breakdown por motivo de falha
        - lista de FAILs (unistrut_tag, beam_tag, motivo, overhang mínimo)
    """
    import ifcopenshell
    from pathlib import Path
    from collections import Counter

    path = Path(ifc_file_path)
    if not path.exists():
        return f"ERRO: Arquivo não encontrado — {ifc_file_path}"

    # ── 1. Abre o modelo ──────────────────────────────────────────────────────
    model = ifcopenshell.open(str(path))
    beams   = model.by_type("IfcBeam")
    proxies = model.by_type("IfcBuildingElementProxy")

    # ── 2. Filtra proxies relevantes ──────────────────────────────────────────
    target_proxies = []
    for proxy in proxies:
        psets = get_psets(proxy)
        kind  = looks_like_target_proxy(proxy, psets)
        if kind:
            row        = summarize_entity(proxy)
            row["kind"]  = kind
            row["psets"] = psets
            target_proxies.append(row)

    unistruts_all  = [x for x in target_proxies if x["kind"] == "UNISTRUT"]
    trapezes_all   = [x for x in target_proxies if x["kind"] == "TRAPEZE"]
    cantilevers_all= [x for x in target_proxies if x["kind"] == "CANTILEVER"]

    # ── 3. Pareamento unistrut × viga ─────────────────────────────────────────
    all_pairs = find_unistrut_beam_pairs_by_projection(
        model,
        z_tol_mm=5.0,
        plan_tol_mm=5.0,
        min_parallel_overlap_mm=100.0,
        cross_tol_mm=100.0,
    )

    orientation_count = Counter(r["orientation"] for r in all_pairs)
    n_perp = orientation_count.get("PERPENDICULAR", 0)
    n_para = orientation_count.get("PARALLEL", 0)

    perpendicular_pairs = [r for r in all_pairs if r["orientation"] == "PERPENDICULAR"]

    # ── 4. Avaliação PASS/FAIL ────────────────────────────────────────────────
    rule_results  = evaluate_pass_fail(perpendicular_pairs, model)
    df_checklist  = build_checklist_dataframe(rule_results)
    total_checks  = len(df_checklist)
    pass_count_r  = int(df_checklist["Rule_Pass"].fillna(False).sum())
    fail_count_r  = total_checks - pass_count_r
    reason_counter= Counter(df_checklist["Rule_Reason"].fillna("NO_REASON"))

    # ── 5. Monta o texto de resposta ──────────────────────────────────────────
    lines = [
        f"=== IFC COMPLIANCE REPORT — {path.name} ===",
        "",
        "── ELEMENTOS NO MODELO ──",
        f"  IfcBeam total          : {len(beams)}",
        f"  IfcBuildingElementProxy: {len(proxies)}",
        f"  → Unistruts            : {len(unistruts_all)}",
        f"  → Trapezes             : {len(trapezes_all)}",
        f"  → Cantilevers          : {len(cantilevers_all)}",
        "",
        "── PAREAMENTO UNISTRUT × VIGA ──",
        f"  Total de pares         : {len(all_pairs)}",
        f"  Perpendiculares        : {n_perp}",
        f"  Paralelos              : {n_para}",
        "",
        "── AVALIAÇÃO DE COMPLIANCE (pares perpendiculares) ──",
        f"  Total verificações     : {total_checks}",
        f"  PASS                   : {pass_count_r}",
        f"  FAIL                   : {fail_count_r}",
        "",
        "── BREAKDOWN POR MOTIVO ──",
    ]
    for reason, qty in reason_counter.most_common():
        lines.append(f"  {reason}: {qty}")

    # Lista detalhada dos FAILs
    fails = df_checklist[df_checklist["Rule_Pass"] == False]
    if not fails.empty:
        lines += ["", "── LISTA DE FAILs ──"]
        for _, row in fails.iterrows():
            lines.append(
                f"  Unistrut {row['Unistrut_Tag']} × Beam {row['Beam_Tag']}"
                f" | MinOverhang={row['Min_Overhang_mm']:.1f} mm"
                f" | {row['Rule_Reason']}"
            )

    return "\n".join(lines)



In [ ]:

# =============================================================================
# 19 — TOOL 02 · Report Generator (HTML)
# =============================================================================
# Chama generate_html_report (já definida em célula anterior do notebook) e devolve
# a confirmação do caminho do arquivo gerado.
# =============================================================================

@tool("ifc_report_generator")
def ifc_report_generator(ifc_file_path: str) -> str:
    """
    Gera o relatório HTML de compliance para o arquivo IFC indicado.
    Executa toda a pipeline internamente (pareamento + avaliação + exportação).

    Parâmetros
    ----------
    ifc_file_path : str
        Caminho absoluto ou relativo para o arquivo .ifc.

    Retorno
    -------
    str
        Caminho do arquivo HTML gerado, ou mensagem de erro.
        Resumo em texto com:
        - breakdown por motivo de falha
        - lista de FAILs (unistrut_tag, beam_tag, motivo, overhang mínimo)
    """
    import ifcopenshell
    from pathlib import Path
    from collections import Counter

    path = Path(ifc_file_path)
    if not path.exists():
        return f"ERRO: Arquivo não encontrado — {ifc_file_path}"

    model  = ifcopenshell.open(str(path))

    all_pairs = find_unistrut_beam_pairs_by_projection(
        model,
        z_tol_mm=5.0,
        plan_tol_mm=5.0,
        min_parallel_overlap_mm=100.0,
        cross_tol_mm=100.0,
    )

    perpendicular_pairs = [r for r in all_pairs if r["orientation"] == "PERPENDICULAR"]
    rule_results  = evaluate_pass_fail(perpendicular_pairs, model)
    df_checklist  = build_checklist_dataframe(rule_results)
    total_checks  = len(df_checklist)
    pass_count_r  = int(df_checklist["Rule_Pass"].fillna(False).sum())
    fail_count_r  = total_checks - pass_count_r
    reason_counter= Counter(df_checklist["Rule_Reason"].fillna("NO_REASON"))

    output_html = generate_html_report(
        ifc_path       = str(path),
        df_checklist   = df_checklist,
        pass_count     = pass_count_r,
        fail_count     = fail_count_r,
        reason_counter = reason_counter,
        all_pairs      = all_pairs,
    )

    return f"Relatório HTML gerado com sucesso: {output_html}"



In [ ]:

# =============================================================================
# 20 — SYSTEM PROMPT + CRIAÇÃO DO AGENTE 01 (Compliance Checker)
# =============================================================================

from crewai import Agent, LLM

# ── LLM via Anthropic ────────────────────────────────────────────────────────
llm_anthropic = LLM(
    model    = "anthropic/claude-sonnet-4-5",   # sintaxe CrewAI para provider
    temperature = 0.2,
)

# ── System prompt (persona) do Agente 01 ────────────────────────────────────
AGENT1_BACKSTORY = """
Você é um engenheiro BIM especialista em verificação de conformidade estrutural.
Sua função é analisar arquivos IFC para identificar suportes (unistruts, trapezes,
cantilevers) e verificar se estão corretamente posicionados em relação às vigas
(IfcBeam) do modelo.

Comportamento:
- Sempre execute a tool `ifc_compliance_checker` antes de emitir qualquer conclusão.
- Se precisar gerar o relatório visual, execute também `ifc_report_generator`.
- Apresente os resultados de forma técnica e objetiva: totais, PASS/FAIL, motivos.
- Seja direto; evite linguagem especulativa.

Contexto técnico:
- Um par unistrut-viga é PASS quando o overhang mínimo está entre 50 mm e 90 mm
  (caso haja overhang nos dois lados), OU quando há vigas em ambos os lados do
  unistrut (caso não haja overhang duplo).
- FAIL indica posicionamento fora dos limites de overhang ou suporte unilateral.

Formato de saída preferencial:
1. Resumo executivo (totais, pass/fail, taxa de compliance).
2. Lista de FAILs com identificadores e motivo.
3. Recomendações de revisão pontual.
"""

agent_compliance = Agent(
    role        = "IFC Compliance Checker",
    goal        = (
        "Analisar o arquivo IFC fornecido, verificar a conformidade dos suportes "
        "em relação às vigas e gerar o relatório HTML de resultados."
    ),
    backstory   = AGENT1_BACKSTORY,
    tools       = [ifc_compliance_checker, ifc_report_generator],
    llm         = llm_anthropic,
    verbose     = True,
    allow_delegation = False,
)

print("✅  Agente 01 (Compliance Checker) criado.")


✅  Agente 01 (Compliance Checker) criado.


In [ ]:
# =============================================================================
# 21 — SYSTEM PROMPT + CRIAÇÃO DO AGENTE 02 (Compliance Assistant)
# =============================================================================

AGENT2_BACKSTORY = """
Você é um assistente de compliance BIM, especializado em comunicar resultados
de análises de conformidade estrutural para diferentes públicos — engenheiros,
coordenadores de projeto e clientes.

Sua responsabilidade é receber os dados produzidos pelo Agente 01 (Compliance
Checker) e apresentá-los no formato que o usuário solicitar:
  - Resumo executivo em prosa para reunião de kickoff
  - Tabela técnica detalhada com IDs e métricas
  - Lista de itens de ação para o time de revisão
  - Resposta a perguntas específicas sobre um unistrut ou viga

Comportamento:
- Nunca invente dados; baseie-se exclusivamente nas informações recebidas do
  Agente 01.
- Adapte o nível de detalhe técnico ao tom da pergunta do usuário.
- Se a pergunta exigir nova execução da análise IFC, delegue ao Agente 01.

Formato padrão de saída (adaptável):
- Contexto breve (1-2 frases)
- Dados principais (lista ou tabela)
- Conclusão ou próximos passos
"""

agent_assistant = Agent(
    role        = "BIM Compliance Assistant",
    goal        = (
        "Receber os resultados da análise IFC e responder às perguntas do usuário "
        "no formato mais adequado ao contexto solicitado."
    ),
    backstory   = AGENT2_BACKSTORY,
    tools       = [],                    # este agente não acessa ferramentas diretamente
    llm         = llm_anthropic,
    verbose     = True,
    allow_delegation = True,             # pode delegar de volta ao Agente 01 se necessário
)

print("✅  Agente 02 (Compliance Assistant) criado.")


✅  Agente 02 (Compliance Assistant) criado.


In [38]:
# =============================================================================
# 22 — FUNÇÃO DE CHAMADA DA CREWAI
# =============================================================================

from crewai import Crew, Task, Process

def run_ifc_crew(user_question: str, ifc_file: ifc_file) -> str:
    """
    Orquestra os dois agentes CrewAI para responder a uma pergunta sobre
    o arquivo IFC.

    Parâmetros
    ----------
    user_question : str
        Pergunta em linguagem natural do usuário.
    ifc_file : str
        Caminho para o arquivo IFC. Default: "MAIC_V4.ifc".

    Retorno
    -------
    str
        Resposta final da Crew.
    """

    # ── Task 01: analisar o IFC e produzir o relatório ───────────────────────
    task_analysis = Task(
        description=(
            f"Analise o arquivo IFC em '{ifc_file}'. "
            "Use a tool `ifc_compliance_checker` para obter os dados de compliance. "
            "Em seguida, use `ifc_report_generator` para gerar o relatório HTML. "
            "Retorne um resumo completo com: totais de elementos, pares encontrados, "
            "PASS/FAIL por motivo e lista detalhada de FAILs."
        ),
        expected_output=(
            "Texto estruturado com: totais de IfcBeam e proxies, "
            "número de pares (perpendicular/paralelo), contagem de PASS e FAIL, "
            "breakdown de motivos e lista de FAILs com unistrut_tag, beam_tag, "
            "min_overhang e rule_reason. Caminho do HTML gerado."
        ),
        agent=agent_compliance,
    )

    # ── Task 02: responder ao usuário ─────────────────────────────────────────
    task_answer = Task(
        description=(
            f"Com base nos dados produzidos pelo Agente 01, responda à seguinte "
            f"pergunta do usuário:\n\n\"{user_question}\"\n\n"
            "Adapte o formato (tabela, lista, prosa) ao que a pergunta exige. "
            "Se a pergunta for sobre um unistrut ou viga específicos, filtre os "
            "dados relevantes. Se pedir resumo executivo, seja conciso e objetivo."
        ),
        expected_output=(
            "Resposta clara e bem formatada à pergunta do usuário, "
            "fundamentada exclusivamente nos dados do Agente 01."
        ),
        agent=agent_assistant,
        context=[task_analysis],          # recebe o output do task anterior
    )

    # ── Crew sequencial ───────────────────────────────────────────────────────
    crew = Crew(
        agents  = [agent_compliance, agent_assistant],
        tasks   = [task_analysis, task_answer],
        process = Process.sequential,
        verbose = True,
    )

    result = crew.kickoff()
    return str(result)



In [39]:
# =============================================================================
# 23 — TESTE: enviando uma pergunta para a Crew
# =============================================================================
# Edite `user_question` e `ifc_file` conforme necessário e execute a célula.
# =============================================================================

print("🚀  Iniciando análise CrewAI...\n")
resposta = run_ifc_crew(user_question=user_question, ifc_file=ifc_file)

print("\n" + "=" * 70)
print("RESPOSTA FINAL DA CREW")
print("=" * 70)
print(resposta)


🚀  Iniciando análise CrewAI...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0d1c201b-2c83-4e6d-ba30-4ad04a7f0d67                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analise o arquivo IFC em 'MAIC_ROOM_02.ifc'. Use a tool `ifc_compliance_checker` para obter os dados de  │
│  compliance. Em seguida, use `ifc_report_generator` para gerar o relatório HTML. Retorne um resumo completo     │
│  com: totais de elementos, pares encontrados, PASS/FAIL por motivo e lista detalhada de FAILs.                  │
│  ID: 4cab8ce2-3b33-4ac6-9e7f-aea61e51c376                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: IFC Compliance Checker                                                                                  │
│                                                                                                                 │
│  Task: Analise o arquivo IFC em 'MAIC_ROOM_02.ifc'. Use a tool `ifc_compliance_checker` para obter os dados de  │
│  compliance. Em seguida, use `ifc_report_generator` para gerar o relatório HTML. Retorne um resumo completo     │
│  com: totais de elementos, pares encontrados, PASS/FAIL por motivo e lista detalhada de FAILs.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: ifc_compliance_checker                                                                                   │
│  Args: {'ifc_file_path': 'MAIC_ROOM_02.ifc'}                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Com base nos dados produzidos pelo Agente 01, responda à seguinte pergunta do usuário:                   │
│                                                                                                                 │
│  "Quais unistruts estão em FAIL e quais são os motivos principais? Apresente em formato de tabela com: Tag do   │
│  Unistrut, Tag da Viga, Overhang Mínimo (mm) e Motivo da Falha,Quantas cantilever estão em falha?Quais as       │
│  salas de cada unistrut?"                                                                                       │
│                                                                                                                 │
│  Adapte o formato (tabela, lista, prosa) ao que a pergunta exige. Se a pergunta for sobre um unistrut ou viga   │
│  específicos, filtre os dados relevantes. Se pedir resumo executivo, seja conciso e objetivo.                   │
│  Agent: BIM Compliance Assistant                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 
RESPOSTA FINAL DA CREW
# 📊 RELATÓRIO DE UNISTRUTS EM FALHA — MAIC_ROOM_02.ifc

---

## **RESUMO EXECUTIVO**

Foram identificados **71 unistruts em situação de FAIL** (79,8% do total de 89 verificações perpendiculares). Os principais motivos de não conformidade são:

- **37 casos (52%)**: Overhang fora da faixa permitida (50-90 mm)
- **34 casos (48%)**: Ausência de overhang sem vigas em ambos os lados

---

## **TABELA DETALHADA DE UNISTRUTS EM FAIL**

| # | Tag Unistrut | Tag Viga | Overhang Mín. (mm) | Motivo da Falha |
|---|--------------|----------|-------------------|-----------------|
| 1 | 8710670 | 6183124 | 0.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |
| 2 | 8712019 | 6183121 | -150.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |
| 3 | 8712029 | 6183129 | -59.6 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |
| 4 | 8712898 | 6183121 | -150.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |
| 5 | 8714416 | 6183129 | -59.



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0d1c201b-2c83-4e6d-ba30-4ad04a7f0d67                                                                       │
│  Final Output: # 📊 RELATÓRIO DE UNISTRUTS EM FALHA — MAIC_ROOM_02.ifc                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## **RESUMO EXECUTIVO**                                                                                        │
│                                                                                                                 │
│  Foram identificados **71 unistruts em situação de FAIL** (79,8% do total de 89 verificações perpendiculares).  │
│  Os principais motivos de não conformidade são:                                                                 │
│                                                                                                                 │
│  - **37 casos (52%)**: Overhang fora da faixa permitida (50-90 mm)                                              │
│  - **34 casos (48%)**: Ausência de overhang sem vigas em ambos os lados                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## **TABELA DETALHADA DE UNISTRUTS EM FAIL**                                                                   │
│                                                                                                                 │
│  | # | Tag Unistrut | Tag Viga | Overhang Mín. (mm) | Motivo da Falha |                                         │
│  |---|--------------|----------|-------------------|-----------------|                                          │
│  | 1 | 8710670 | 6183124 | 0.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |                                           │
│  | 2 | 8712019 | 6183121 | -150.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                 │
│  | 3 | 8712029 | 6183129 | -59.6 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                  │
│  | 4 | 8712898 | 6183121 | -150.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                 │
│  | 5 | 8714416 | 6183129 | -59.6 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                  │
│  | 6 | 8830176 | 6241976 | -100.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                 │
│  | 7 | 8830178 | 6241976 | -100.0 | NO_OVERHANG_AND_NOT_BEAMS_BOTH_SIDES_FAIL |                                 │
│  | 8 | 8911625 | 4999222 | 100.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |                                         │
│  | 9 | 8912160 | 4999222 | 100.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |                                         │
│  | 10 | 8912371 | 4999222 | 100.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |                                        │
│  | 11 | 8912435 | 4999222 | 100.0 | OVERHANG_MIN_OUTSIDE_50_TO_90_FAIL |                                        │
│  | 12 | 8912769 | 4999197 | 208.7 | OVERHANG_MIN_OUTSID